# Notebook 4: Building a Simple VLA Model

A **Vision-Language-Action (VLA)** model takes visual observations and language instructions as input and predicts robot actions. In this notebook you will build a minimal VLA action head that mirrors the architecture in the SimVLA codebase.

You will implement:

1. **Action Encoder / Decoder** — project (action + proprio + time) to hidden dim
2. **Concat-Mode Forward** — concatenate action tokens with VLM features
3. **AdaLN-Mode Forward** — inject conditioning via adaptive layer norm
4. **Flow Matching Training Loop** — end-to-end training on synthetic data
5. **Action Generation via Euler Integration** — inference pipeline

We use simulated VLM features (random tensors) so no GPU or pretrained model is needed.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
# ====== Provided helper modules ======

def timestep_embedding(t: torch.Tensor, dim: int, max_period: int = 100) -> torch.Tensor:
    """Sinusoidal timestep embedding. [B] -> [B, dim]"""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=t.dtype, device=t.device) / half)
    args = t[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
    return emb


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj_drop(self.proj(x))


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, drop=0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate='tanh')
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.attn = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, attn_drop=0.1)
        self.mlp = Mlp(hidden_size, int(hidden_size * mlp_ratio), drop=0.1)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, attn_drop=0.1)
        self.mlp = Mlp(hidden_size, int(hidden_size * mlp_ratio), drop=0.1)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        params = self.adaLN_modulation(c)
        s_msa, sc_msa, g_msa, s_mlp, sc_mlp, g_mlp = params.chunk(6, dim=-1)
        x = x + g_msa.unsqueeze(1) * self.attn(modulate(self.norm1(x), s_msa, sc_msa))
        x = x + g_mlp.unsqueeze(1) * self.mlp(modulate(self.norm2(x), s_mlp, sc_mlp))
        return x


class FinalLayer(nn.Module):
    def __init__(self, hidden_size, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(hidden_size, 2 * hidden_size, bias=True)
        )
        self.linear = nn.Linear(hidden_size, out_dim, bias=True)
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.linear.weight, 0)
        nn.init.constant_(self.linear.bias, 0)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=-1)
        return self.linear(modulate(self.norm(x), shift, scale))


def basic_init(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)

---
## Exercise 4a: Action Encoder & Decoder

In **concat mode**, the action encoder takes the concatenation of:
- noisy action tokens `[B, T_action, dim_action]`
- proprioceptive state (broadcast) `[B, T_action, dim_proprio]`
- timestep embedding (broadcast) `[B, T_action, dim_time]`

And projects to hidden dimension:
$$\text{tokens} = W_{\text{enc}} \cdot [a_t \| s \| t_{\text{emb}}] + b_{\text{enc}}$$

The decoder maps back from hidden dim to action dim:
$$\hat{v} = W_{\text{dec}} \cdot h + b_{\text{dec}}$$

In [ ]:
def build_action_encoder_decoder(
    dim_action: int,
    dim_proprio: int,
    dim_time: int,
    hidden_size: int,
) -> tuple:
    """
    Build a (concat-mode) action encoder and decoder.

    Returns
    -------
    encoder : nn.Linear  mapping (dim_action + dim_proprio + dim_time) -> hidden_size
    decoder : nn.Linear  mapping hidden_size -> dim_action
    """
    # TODO:
    # 1. input_dim = dim_action + dim_proprio + dim_time
    # 2. encoder = nn.Linear(input_dim, hidden_size)
    # 3. decoder = nn.Linear(hidden_size, dim_action)
    # 4. Return (encoder, decoder)
    raise NotImplementedError("Implement build_action_encoder_decoder")

In [ ]:
# === Tests for Exercise 4a ===
dim_action, dim_proprio, dim_time, hidden_size = 7, 8, 32, 128
enc, dec = build_action_encoder_decoder(dim_action, dim_proprio, dim_time, hidden_size)

assert isinstance(enc, nn.Linear), "Encoder should be nn.Linear"
assert isinstance(dec, nn.Linear), "Decoder should be nn.Linear"
assert enc.in_features == dim_action + dim_proprio + dim_time
assert enc.out_features == hidden_size
assert dec.in_features == hidden_size
assert dec.out_features == dim_action

# Functional test
B, T_act = 2, 30
a = torch.randn(B, T_act, dim_action)
p = torch.randn(B, dim_proprio).unsqueeze(1).expand(B, T_act, dim_proprio)
t_emb = torch.randn(B, dim_time).unsqueeze(1).expand(B, T_act, dim_time)
cat_input = torch.cat([a, p, t_emb], dim=-1)
tokens = enc(cat_input)
assert tokens.shape == (B, T_act, hidden_size)

output = dec(tokens)
assert output.shape == (B, T_act, dim_action)

print("Exercise 4a passed!")

---
## Exercise 4b: Concat-Mode Action Transformer Forward

The concat-mode forward pass:

1. Encode `[action, proprio, time]` into tokens
2. Project VLM features and **concatenate** with action tokens
3. Add learned positional embeddings
4. Process through transformer blocks
5. Decode only the first `T_action` tokens back to action dim

This is the architecture used when `use_adaln=False` in the SimVLA codebase.

In [ ]:
class ConcatActionTransformer(nn.Module):
    """
    Action Transformer in concat mode.

    Parameters
    ----------
    hidden_size : int
    vlm_hidden_size : int -- dim of VLM features
    depth : int -- number of transformer blocks
    num_heads : int
    mlp_ratio : float
    dim_action : int
    dim_proprio : int
    dim_time : int
    max_len_seq : int -- max total sequence length
    """

    def __init__(
        self,
        hidden_size: int = 128,
        vlm_hidden_size: int = 64,
        depth: int = 4,
        num_heads: int = 4,
        mlp_ratio: float = 4.0,
        dim_action: int = 7,
        dim_proprio: int = 8,
        dim_time: int = 32,
        max_len_seq: int = 512,
    ):
        super().__init__()
        self.dim_action = dim_action
        self.dim_time = dim_time

        # TODO: Define the following:
        # self.blocks -> nn.ModuleList of `depth` TransformerBlock(hidden_size, num_heads, mlp_ratio)
        # self.vlm_proj -> nn.Linear(vlm_hidden_size, hidden_size)
        # self.pos_emb -> nn.Parameter(zeros(1, max_len_seq, hidden_size)), init with normal(std=0.02)
        # self.norm -> nn.LayerNorm(hidden_size)
        # self.action_encoder -> nn.Linear(dim_action + dim_proprio + dim_time, hidden_size)
        # self.action_decoder -> nn.Linear(hidden_size, dim_action)
        # Apply basic_init to all modules: self.apply(basic_init)
        raise NotImplementedError("Define layers")

    def forward(
        self,
        vlm_features: torch.Tensor,       # [B, T_vlm, vlm_hidden_size]
        action_with_noise: torch.Tensor,  # [B, T_action, dim_action]
        proprio: torch.Tensor,            # [B, dim_proprio]
        t: torch.Tensor,                  # [B] scalar timesteps
    ) -> torch.Tensor:
        """
        Returns predicted velocity [B, T_action, dim_action].
        """
        B, num_actions = action_with_noise.shape[:2]

        # TODO:
        # 1. Time embedding: t_emb = timestep_embedding(t, self.dim_time) -> [B, dim_time]
        #    Expand to [B, num_actions, dim_time]
        # 2. Proprio: expand to [B, num_actions, dim_proprio]
        # 3. Concatenate: [action_with_noise, proprio, t_emb] along dim=-1
        # 4. Encode: x = self.action_encoder(concatenated) -> [B, num_actions, hidden_size]
        # 5. Project VLM: vlm_proj = self.vlm_proj(vlm_features) -> [B, T_vlm, hidden_size]
        # 6. Concatenate: x = cat([x, vlm_proj], dim=1) -> [B, num_actions + T_vlm, hidden_size]
        # 7. Add positional embedding: x = x + self.pos_emb[:, :seq_len, :]
        # 8. Process through self.blocks
        # 9. Decode action segment: self.action_decoder(self.norm(x[:, :num_actions]))
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 4b ===
torch.manual_seed(100)
B, T_vlm, T_act = 2, 20, 30
dim_action, dim_proprio = 7, 8
vlm_hidden = 64
hidden = 128

model_concat = ConcatActionTransformer(
    hidden_size=hidden, vlm_hidden_size=vlm_hidden,
    depth=2, num_heads=4, dim_action=dim_action, dim_proprio=dim_proprio,
).eval()

vlm_feat = torch.randn(B, T_vlm, vlm_hidden)
actions = torch.randn(B, T_act, dim_action)
proprio = torch.randn(B, dim_proprio)
t = torch.rand(B)

out = model_concat(vlm_feat, actions, proprio, t)

# Shape
assert out.shape == (B, T_act, dim_action), f"Expected {(B, T_act, dim_action)}, got {out.shape}"

# Deterministic in eval
out2 = model_concat(vlm_feat, actions, proprio, t)
assert torch.allclose(out, out2, atol=1e-5), "Should be deterministic in eval mode"

# Gradient flow
model_concat.train()
actions_g = torch.randn(B, T_act, dim_action, requires_grad=True)
loss = model_concat(vlm_feat, actions_g, proprio, t).sum()
loss.backward()
assert actions_g.grad is not None, "Gradients must flow through actions"

print("Exercise 4b passed!")

---
## Exercise 4c: AdaLN-Mode Action Transformer Forward

In AdaLN mode, conditioning (time, VLM, proprio) is injected via adaptive layer normalization rather than concatenation:

1. **Build condition:** $c = t_{\text{emb}} + \text{VLM}_{\text{pool}} + \text{proprio}_{\text{proj}}$
   - Time: sinusoidal embedding projected to hidden_size
   - VLM: global average pooling of VLM features, then linear projection
   - Proprio: linear projection
2. Encode only action tokens
3. Process through DiT blocks with condition $c$
4. Final layer with AdaLN

In [ ]:
class AdaLNActionTransformer(nn.Module):
    """
    Action Transformer in AdaLN (DiT) mode.
    """

    def __init__(
        self,
        hidden_size: int = 128,
        vlm_hidden_size: int = 64,
        depth: int = 4,
        num_heads: int = 4,
        mlp_ratio: float = 4.0,
        dim_action: int = 7,
        dim_proprio: int = 8,
        max_len_seq: int = 512,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.dim_action = dim_action

        # TODO: Define:
        # self.blocks -> ModuleList of `depth` DiTBlock(hidden_size, num_heads, mlp_ratio)
        #
        # Condition encoders:
        # self.time_proj -> Sequential(Linear(hidden_size, hidden_size), SiLU(), Linear(hidden_size, hidden_size))
        # self.vlm_cond_proj -> Linear(vlm_hidden_size, hidden_size)
        # self.proprio_proj -> Linear(dim_proprio, hidden_size)
        #
        # Action encoder (action-only, no concat with time/proprio):
        # self.action_encoder -> Linear(dim_action, hidden_size)
        #
        # Positional embedding:
        # self.pos_emb -> Parameter(zeros(1, max_len_seq, hidden_size)), init normal(std=0.02)
        #
        # Final layer:
        # self.final_layer -> FinalLayer(hidden_size, dim_action)
        #
        # self.apply(basic_init)
        raise NotImplementedError("Define layers")

    def forward(
        self,
        vlm_features: torch.Tensor,       # [B, T_vlm, vlm_hidden_size]
        action_with_noise: torch.Tensor,  # [B, T_action, dim_action]
        proprio: torch.Tensor,            # [B, dim_proprio]
        t: torch.Tensor,                  # [B]
    ) -> torch.Tensor:
        """
        Returns predicted velocity [B, T_action, dim_action].
        """
        B, num_actions = action_with_noise.shape[:2]

        # TODO:
        # 1. Time condition: t_emb = timestep_embedding(t, self.hidden_size)
        #    t_emb = self.time_proj(t_emb)  -> [B, hidden_size]
        #
        # 2. VLM condition: global average pool over sequence dim, then project
        #    vlm_cond = self.vlm_cond_proj(vlm_features.mean(dim=1))  -> [B, hidden_size]
        #
        # 3. Proprio condition: self.proprio_proj(proprio)  -> [B, hidden_size]
        #
        # 4. Fuse: c = t_emb + vlm_cond + proprio_cond  -> [B, hidden_size]
        #
        # 5. Encode actions: x = self.action_encoder(action_with_noise)  -> [B, T_action, hidden_size]
        # 6. Add pos_emb: x = x + self.pos_emb[:, :num_actions, :]
        #
        # 7. DiT blocks: for block in self.blocks: x = block(x, c)
        #
        # 8. Final layer: return self.final_layer(x, c)
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 4c ===
torch.manual_seed(101)
B, T_vlm, T_act = 2, 20, 30
dim_action, dim_proprio = 7, 8
vlm_hidden = 64
hidden = 128

model_adaln = AdaLNActionTransformer(
    hidden_size=hidden, vlm_hidden_size=vlm_hidden,
    depth=2, num_heads=4, dim_action=dim_action, dim_proprio=dim_proprio,
).eval()

vlm_feat = torch.randn(B, T_vlm, vlm_hidden)
actions = torch.randn(B, T_act, dim_action)
proprio = torch.randn(B, dim_proprio)
t = torch.rand(B)

out = model_adaln(vlm_feat, actions, proprio, t)

# Shape must match concat mode
assert out.shape == (B, T_act, dim_action), f"Expected {(B, T_act, dim_action)}, got {out.shape}"

# Deterministic
out2 = model_adaln(vlm_feat, actions, proprio, t)
assert torch.allclose(out, out2, atol=1e-5)

# Gradient flow through conditioning
model_adaln.train()
proprio_g = torch.randn(B, dim_proprio, requires_grad=True)
loss = model_adaln(vlm_feat, actions, proprio_g, t).sum()
loss.backward()
assert proprio_g.grad is not None, "Gradients must flow through proprio conditioning"

print("Exercise 4c passed!")

---
## Exercise 4d: Flow Matching Training Loop

Wire the action transformer into a complete flow matching training loop:

1. Sample $t \sim \text{Beta}(1.5, 1) \cdot 0.999 + 0.001$
2. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
3. Interpolate: $x_t = t \cdot \epsilon + (1-t) \cdot a$
4. Target velocity: $u_t = \epsilon - a$
5. Predict: $v_\theta = \text{model}(\text{vlm}, x_t, s, t)$
6. Loss: $\text{MSE}(v_\theta, u_t)$

In [ ]:
def vla_flow_matching_train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    vlm_features: torch.Tensor,   # [B, T_vlm, D_vlm]
    actions: torch.Tensor,        # [B, T_action, dim_action] — ground truth
    proprio: torch.Tensor,        # [B, dim_proprio]
) -> float:
    """
    One training step for the VLA action transformer with flow matching.

    Returns
    -------
    loss_value : float
    """
    B = actions.shape[0]

    # TODO:
    # 1. Sample t ~ Beta(1.5, 1.0) for each sample, scale: t = t * 0.999 + 0.001
    # 2. Sample noise ~ N(0, I) same shape as actions
    # 3. Reshape t to [B, 1, 1] for broadcasting with [B, T_action, dim_action]
    # 4. Interpolate: x_t = t_expanded * noise + (1 - t_expanded) * actions
    # 5. Target velocity: u_t = noise - actions
    # 6. Predict: v_pred = model(vlm_features, x_t, proprio, t)  (t is [B])
    # 7. Loss = mean((v_pred - u_t)^2)
    # 8. optimizer.zero_grad(), loss.backward(), optimizer.step()
    # 9. Return loss.item()
    raise NotImplementedError("Implement vla_flow_matching_train_step")

In [ ]:
# === Tests for Exercise 4d ===
torch.manual_seed(200)
B, T_vlm, T_act = 8, 10, 15
dim_action, dim_proprio = 7, 8
vlm_hidden, hidden = 64, 128

model = ConcatActionTransformer(
    hidden_size=hidden, vlm_hidden_size=vlm_hidden, depth=2, num_heads=4,
    dim_action=dim_action, dim_proprio=dim_proprio,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Synthetic training data: sine wave actions
time_axis = torch.linspace(0, 2 * math.pi, T_act).unsqueeze(0).unsqueeze(-1)
gt_actions = torch.sin(time_axis).expand(B, T_act, dim_action) + 0.1 * torch.randn(B, T_act, dim_action)
vlm_feat = torch.randn(B, T_vlm, vlm_hidden)  # simulated VLM features
proprio_data = torch.randn(B, dim_proprio)

# Train
model.train()
losses = []
for step in range(100):
    loss = vla_flow_matching_train_step(model, optimizer, vlm_feat, gt_actions, proprio_data)
    losses.append(loss)

assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"

# Loss should be finite
assert all(math.isfinite(l) for l in losses), "All losses should be finite"

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print("Exercise 4d passed!")

---
## Exercise 4e: Action Generation via Euler Integration

At inference time, generate actions by integrating the velocity field from $t=1$ (noise) to $t=0$ (actions):

$$x_t \leftarrow x_t + \Delta t \cdot v_\theta(\text{vlm}, x_t, s, t)$$

where $\Delta t = -1/N$ and we step from $t=1$ to $t=0$.

In [ ]:
@torch.no_grad()
def generate_actions(
    model: nn.Module,
    vlm_features: torch.Tensor,  # [B, T_vlm, D_vlm]
    proprio: torch.Tensor,       # [B, dim_proprio]
    num_actions: int,            # T_action
    dim_action: int,
    num_steps: int = 10,
) -> torch.Tensor:
    """
    Generate actions via Euler integration of the flow matching velocity field.

    Returns
    -------
    actions : Tensor [B, num_actions, dim_action]
    """
    B = vlm_features.shape[0]

    # TODO:
    # 1. Initialize x = randn(B, num_actions, dim_action)
    # 2. dt = -1.0 / num_steps
    # 3. t_current = 1.0
    # 4. For _ in range(num_steps):
    #      a. t_batch = tensor of shape [B] filled with t_current
    #      b. v = model(vlm_features, x, proprio, t_batch)
    #      c. x = x + dt * v
    #      d. t_current = t_current + dt
    # 5. Return x
    raise NotImplementedError("Implement generate_actions")

In [ ]:
# === Tests for Exercise 4e ===
torch.manual_seed(300)
model.eval()

gen_actions = generate_actions(
    model, vlm_feat[:2], proprio_data[:2],
    num_actions=T_act, dim_action=dim_action, num_steps=20,
)

# Shape
assert gen_actions.shape == (2, T_act, dim_action), f"Expected (2, {T_act}, {dim_action}), got {gen_actions.shape}"

# Should be finite
assert torch.isfinite(gen_actions).all(), "Generated actions should be finite"

# Different VLM features should produce different actions
vlm_a = torch.randn(1, T_vlm, vlm_hidden)
vlm_b = torch.randn(1, T_vlm, vlm_hidden)
p = torch.randn(1, dim_proprio)
actions_a = generate_actions(model, vlm_a, p, T_act, dim_action, num_steps=10)
actions_b = generate_actions(model, vlm_b, p, T_act, dim_action, num_steps=10)
assert not torch.allclose(actions_a, actions_b, atol=1e-3), "Different VLM features -> different actions"

# More steps should be valid
actions_many = generate_actions(model, vlm_a, p, T_act, dim_action, num_steps=50)
assert actions_many.shape == (1, T_act, dim_action)

print("Exercise 4e passed!")
print("\n=== All Notebook 4 exercises passed! ===")